# Week 8 Dataset Validation

**Student:** Manthan Parmar  
**Organization:** CS Vision  
**Internship:** Financial & Operational Analytics Internship Program  
**Week:** 8 – Final Project & Presentation

This notebook validates the supplied cleaned and date-fixed CSV snapshots without changing either source file. It uses pandas and NumPy to check structure, quality, dates, financial calculations, and consistency with the Week 1 preprocessing process. Run the cells from top to bottom.

## 1. Introduction

The validation treats `financial_retail_clean.csv` and `financial_retail_fixed.csv` as separate read-only inputs. The stated expectation is 850 transactions and 19 columns in each file; the second file is expected to standardize date formatting, not add records.

## 2. Load Datasets

The project root is found by searching the current folder and its parents. The raw Week 1 source is loaded only to verify the preprocessing notebook's `TXN100505` assumption.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'data' / 'cleaned' / 'financial_retail_clean.csv').is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Could not locate the project root containing data/cleaned/financial_retail_clean.csv.')

clean_path = project_root / 'data' / 'cleaned' / 'financial_retail_clean.csv'
fixed_path = project_root / 'data' / 'cleaned' / 'financial_retail_fixed.csv'
raw_path = project_root / 'data' / 'raw' / 'financial_retail_raw.csv'
week1_notebook_path = project_root / 'notebooks' / 'week1_preprocessing' / 'financial_retail_preprocessing.ipynb'
for path in [clean_path, fixed_path, raw_path, week1_notebook_path]:
    if not path.is_file():
        raise FileNotFoundError(f'Required validation input was not found: {path}')

clean_df = pd.read_csv(clean_path)
fixed_df = pd.read_csv(fixed_path)
raw_df = pd.read_csv(raw_path)

results = []

def add_result(category, check, expected, actual, status, remarks=''):
    results.append({
        'Validation Category': category,
        'Validation Check': check,
        'Expected Result': expected,
        'Actual Result': str(actual),
        'Status': status,
        'Remarks': remarks,
    })

expected_columns = [
    'Transaction_ID', 'Date', 'Revenue', 'Cost', 'Operating_Expense',
    'Gross_Profit', 'Net_Profit', 'Product', 'Category', 'Customer_ID',
    'Customer_Segment', 'Region', 'City', 'Quantity', 'Order_Status',
    'Month', 'Quarter', 'Year', 'Profit_Status',
]
datasets = {'Cleaned': clean_df, 'Date-Fixed': fixed_df}
print('Loaded read-only CSV inputs:')
display(pd.DataFrame([
    {'Dataset': name, 'File': path.name, 'Rows': len(frame), 'Columns': frame.shape[1]}
    for name, frame, path in [
        ('Cleaned', clean_df, clean_path), ('Date-Fixed', fixed_df, fixed_path)
    ]
]))

Loaded read-only CSV inputs:


,Dataset,File,Rows,Columns
0,Cleaned,financial_retail_clean.csv,850,19
1,Date-Fixed,financial_retail_fixed.csv,850,19


## 3. Dataset Overview

Review dimensions, column order, sample records, and data types. The expected column sequence is the schema used by the Week 1 cleaned output.

In [2]:
overview = []
for name, frame in datasets.items():
    overview.append({
        'Dataset': name,
        'Rows': len(frame),
        'Columns': frame.shape[1],
        'Unique transactions': frame['Transaction_ID'].nunique(),
        'Exact duplicate rows': int(frame.duplicated().sum()),
    })
    shape_ok = frame.shape == (850, 19)
    add_result('Structure', f'{name}: row and column count', '850 rows and 19 columns', frame.shape, 'PASS' if shape_ok else 'FAIL')
    order_ok = frame.columns.tolist() == expected_columns
    add_result('Structure', f'{name}: column names and order', expected_columns, frame.columns.tolist(), 'PASS' if order_ok else 'FAIL')

display(pd.DataFrame(overview))
display(clean_df.head())

,Dataset,Rows,Columns,Unique transactions,Exact duplicate rows
0,Cleaned,850,19,850,0
1,Date-Fixed,850,19,850,0


,Transaction_ID,Date,Revenue,Cost,Operating_Expense,Gross_Profit,Net_Profit,Product,Category,Customer_ID,Customer_Segment,Region,City,Quantity,Order_Status,Month,Quarter,Year,Profit_Status
0,TXN100447,1/1/2025,58470.21,36725.89,7621.28,21744.32,14123.04,Monitor,Electronics,CUST1087,Consumer,South,Chennai,4,Completed,January,Q1,2025,Profit
1,TXN100678,1/1/2025,24860.80,16154.15,3382.73,8706.65,5323.92,Smartphone,Electronics,CUST1189,Consumer,North,Delhi,1,Completed,January,Q1,2025,Profit
2,TXN100649,1/1/2025,29366.04,18343.70,5027.60,11022.34,5994.74,Smartphone,Electronics,CUST1163,Consumer,South,Chennai,1,Completed,January,Q1,2025,Profit
3,TXN100756,1/1/2025,8456.55,4776.35,1291.80,3680.20,2388.40,Smartwatch,Accessories,CUST1072,Corporate,West,Ahmedabad,1,Cancelled,January,Q1,2025,Profit
4,TXN100849,1/1/2025,68559.66,41428.90,12170.47,27130.76,14960.29,Laptop,Electronics,CUST1161,Consumer,North,Delhi,1,Completed,January,Q1,2025,Profit


## 4. Missing Value Validation

Missing values are counted directly. Explicit `Unknown`/`UNKNOWN` labels are reported separately because Week 1 preprocessing used them as placeholders; they are not treated as null cells.

In [3]:
missing_table = pd.DataFrame({name: frame.isna().sum() for name, frame in datasets.items()})
display(missing_table)
for name, frame in datasets.items():
    missing_count = int(frame.isna().sum().sum())
    add_result('Missing Values', f'{name}: missing cells', '0 missing cells', missing_count, 'PASS' if missing_count == 0 else 'FAIL')

placeholder_counts = {
    'Product = Unknown': int(clean_df['Product'].eq('Unknown').sum()),
    'Customer_ID = UNKNOWN': int(clean_df['Customer_ID'].eq('UNKNOWN').sum()),
    'City = Unknown': int(clean_df['City'].eq('Unknown').sum()),
    'Order_Status = Unknown': int(clean_df['Order_Status'].eq('Unknown').sum()),
}
display(pd.Series(placeholder_counts, name='Rows with explicit placeholder'))
placeholder_total = sum(placeholder_counts.values())
add_result('Missing Values', 'Explicit Week 1 placeholder labels', 'Report separately; not null cells', placeholder_counts, 'WARNING' if placeholder_total else 'PASS', 'Placeholder counts can overlap only if a row has more than one such label.')

,Cleaned,Date-Fixed
Transaction_ID,0,0
Date,0,0
Revenue,0,0
Cost,0,0
Operating_Expense,0,0
Gross_Profit,0,0
Net_Profit,0,0
Product,0,0
Category,0,0
Customer_ID,0,0


Product = Unknown          7
Customer_ID = UNKNOWN     18
City = Unknown            10
Order_Status = Unknown     8
Name: Rows with explicit placeholder, dtype: int64

## 5. Duplicate Validation

Check exact duplicate rows, repeated transaction identifiers, and the number of unique transaction IDs in each output.

In [4]:
duplicate_table = []
for name, frame in datasets.items():
    exact_duplicates = int(frame.duplicated().sum())
    duplicate_ids = int(frame['Transaction_ID'].duplicated().sum())
    unique_ids = int(frame['Transaction_ID'].nunique())
    duplicate_table.append({
        'Dataset': name,
        'Exact duplicate rows': exact_duplicates,
        'Duplicate Transaction_ID rows': duplicate_ids,
        'Unique transactions': unique_ids,
    })
    add_result('Duplicates', f'{name}: exact duplicate rows', '0', exact_duplicates, 'PASS' if exact_duplicates == 0 else 'FAIL')
    add_result('Duplicates', f'{name}: duplicate Transaction_ID rows', '0', duplicate_ids, 'PASS' if duplicate_ids == 0 else 'FAIL')
    add_result('Duplicates', f'{name}: unique transaction count', '850', unique_ids, 'PASS' if unique_ids == 850 else 'FAIL')

display(pd.DataFrame(duplicate_table))

,Dataset,Exact duplicate rows,Duplicate Transaction_ID rows,Unique transactions
0,Cleaned,0,0,850
1,Date-Fixed,0,0,850


## 6. Data Type Validation

CSV dates should load as text before parsing. Financial and quantity fields should be numeric. The two files should also have matching pandas-inferred data types.

In [5]:
dtype_table = pd.DataFrame({name: frame.dtypes.astype(str) for name, frame in datasets.items()})
display(dtype_table)
dtypes_match = clean_df.dtypes.astype(str).equals(fixed_df.dtypes.astype(str))
add_result('Data Types', 'Matching inferred types between CSVs', 'Same inferred dtype for every column', dtypes_match, 'PASS' if dtypes_match else 'FAIL')

numeric_columns = ['Revenue', 'Cost', 'Operating_Expense', 'Gross_Profit', 'Net_Profit', 'Quantity', 'Year']
for name, frame in datasets.items():
    numeric_ok = all(pd.api.types.is_numeric_dtype(frame[column]) for column in numeric_columns)
    date_ok = pd.api.types.is_string_dtype(frame['Date'])
    add_result('Data Types', f'{name}: numeric columns', f'Numeric: {numeric_columns}', {column: str(frame[column].dtype) for column in numeric_columns}, 'PASS' if numeric_ok else 'FAIL')
    add_result('Data Types', f'{name}: Date is stored as text before parsing', 'Text/string', str(frame['Date'].dtype), 'PASS' if date_ok else 'FAIL')

,Cleaned,Date-Fixed
Transaction_ID,str,str
Date,str,str
Revenue,float64,float64
Cost,float64,float64
Operating_Expense,float64,float64
Gross_Profit,float64,float64
Net_Profit,float64,float64
Product,str,str
Category,str,str
Customer_ID,str,str


## 7. Date Validation

The cleaned CSV uses US-style `M/D/YYYY` date strings and the date-fixed CSV uses ISO `YYYY-MM-DD`. Parse both with the same month-first interpretation, check invalid dates, compare by transaction ID, and independently recompute Month, Quarter, and Year.

In [6]:
clean_dates = pd.to_datetime(clean_df['Date'], format='mixed', errors='coerce', dayfirst=False)
fixed_dates = pd.to_datetime(fixed_df['Date'], format='mixed', errors='coerce', dayfirst=False)
clean_us_format = int(clean_df['Date'].astype('string').str.fullmatch(r'\d{1,2}/\d{1,2}/\d{4}').sum())
fixed_iso_format = int(fixed_df['Date'].astype('string').str.fullmatch(r'\d{4}-\d{2}-\d{2}').sum())
date_format_table = pd.DataFrame([
    {'Dataset': 'Cleaned', 'Observed format': 'M/D/YYYY', 'Matching strings': clean_us_format, 'Unparseable dates': int(clean_dates.isna().sum())},
    {'Dataset': 'Date-Fixed', 'Observed format': 'YYYY-MM-DD (ISO)', 'Matching strings': fixed_iso_format, 'Unparseable dates': int(fixed_dates.isna().sum())},
])
display(date_format_table)

for name, count, dates in [
    ('Cleaned', clean_us_format, clean_dates), ('Date-Fixed', fixed_iso_format, fixed_dates)
]:
    expected_format_count = 850
    add_result('Dates', f'{name}: observed date format', f'{expected_format_count} dates in expected format', count, 'PASS' if count == expected_format_count else 'FAIL')
    invalid_count = int(dates.isna().sum())
    add_result('Dates', f'{name}: invalid dates', '0 invalid dates', invalid_count, 'PASS' if invalid_count == 0 else 'FAIL')

clean_by_id = pd.Series(clean_dates.to_numpy(), index=clean_df['Transaction_ID']).sort_index()
fixed_by_id = pd.Series(fixed_dates.to_numpy(), index=fixed_df['Transaction_ID']).sort_index()
same_ids = clean_by_id.index.equals(fixed_by_id.index)
matching_dates = int(clean_by_id.eq(fixed_by_id).sum()) if same_ids else 0
add_result('Dates', 'Transaction dates match after datetime standardization', '850 matching dates', f'{matching_dates} matching dates', 'PASS' if same_ids and matching_dates == 850 else 'FAIL')
raw_date_differences = int(clean_df['Date'].ne(fixed_df['Date']).sum())
add_result('Dates', 'Raw date strings differ due to formatting', 'Formatting may differ; parsed dates must match', f'{raw_date_differences} of 850 strings differ', 'PASS' if raw_date_differences == 850 and matching_dates == 850 else 'WARNING', 'Raw string differences are expected for the date-standardized file.')

date_label_checks = []
for name, frame, dates in [('Cleaned', clean_df, clean_dates), ('Date-Fixed', fixed_df, fixed_dates)]:
    month_mismatches = int(dates.dt.month_name().ne(frame['Month']).sum())
    quarter_mismatches = int(('Q' + dates.dt.quarter.astype('string')).ne(frame['Quarter']).sum())
    year_mismatches = int(dates.dt.year.ne(frame['Year']).sum())
    date_label_checks.append({'Dataset': name, 'Month mismatches': month_mismatches, 'Quarter mismatches': quarter_mismatches, 'Year mismatches': year_mismatches})
    for label, mismatches in [('Month', month_mismatches), ('Quarter', quarter_mismatches), ('Year', year_mismatches)]:
        add_result('Dates', f'{name}: {label} agrees with Date', '0 mismatches', mismatches, 'PASS' if mismatches == 0 else 'FAIL')
display(pd.DataFrame(date_label_checks))

,Dataset,Observed format,Matching strings,Unparseable dates
0,Cleaned,M/D/YYYY,850,0
1,Date-Fixed,YYYY-MM-DD (ISO),850,0


,Dataset,Month mismatches,Quarter mismatches,Year mismatches
0,Cleaned,0,0,0
1,Date-Fixed,0,0,0


## 8. Financial Calculation Validation

Recompute Gross Profit and Net Profit independently with NumPy. A one-cent tolerance accommodates decimal representation. Negative amounts are reported as warnings rather than automatically treated as errors. Potential outliers use the 1.5×IQR screening rule and require business review; they are not automatically errors.

In [7]:
financial_columns = ['Revenue', 'Cost', 'Operating_Expense', 'Gross_Profit', 'Net_Profit']
financial_summary = []
iqr_counts = {}
iqr_flagged_rows = pd.Series(False, index=clean_df.index)

for name, frame in datasets.items():
    expected_gross = frame['Revenue'].to_numpy() - frame['Cost'].to_numpy()
    expected_net = frame['Gross_Profit'].to_numpy() - frame['Operating_Expense'].to_numpy()
    gross_mismatches = int((~np.isclose(frame['Gross_Profit'].to_numpy(), expected_gross, atol=0.01, rtol=0)).sum())
    net_mismatches = int((~np.isclose(frame['Net_Profit'].to_numpy(), expected_net, atol=0.01, rtol=0)).sum())
    add_result('Financial Calculations', f'{name}: Gross Profit formula', 'Gross_Profit = Revenue - Cost; 0 mismatches', gross_mismatches, 'PASS' if gross_mismatches == 0 else 'FAIL')
    add_result('Financial Calculations', f'{name}: Net Profit formula', 'Net_Profit = Gross_Profit - Operating_Expense; 0 mismatches', net_mismatches, 'PASS' if net_mismatches == 0 else 'FAIL')

    negative_count = int((frame[financial_columns] < 0).sum().sum())
    negative_status = 'WARNING' if negative_count else 'PASS'
    add_result('Financial Calculations', f'{name}: negative financial values', 'Report count; negatives are not automatically errors', negative_count, negative_status, 'Review any negative values in business context.')
    expected_status = np.where(frame['Net_Profit'].to_numpy() >= 0, 'Profit', 'Loss')
    status_mismatches = int((frame['Profit_Status'].to_numpy() != expected_status).sum())
    add_result('Financial Calculations', f'{name}: Profit_Status consistency', 'Profit for Net_Profit >= 0; otherwise Loss', status_mismatches, 'PASS' if status_mismatches == 0 else 'FAIL')

    for column in financial_columns:
        financial_summary.append({'Dataset': name, 'Measure': column, 'Total': round(float(frame[column].sum()), 2)})

    if name == 'Cleaned':
        for column in financial_columns + ['Quantity']:
            q1 = frame[column].quantile(0.25)
            q3 = frame[column].quantile(0.75)
            iqr = q3 - q1
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            mask = (frame[column] < lower) | (frame[column] > upper)
            count = int(mask.sum())
            iqr_counts[column] = count
            iqr_flagged_rows |= mask
            add_result('Suspicious Values', f'{column}: 1.5xIQR flags', 'Report possible outliers for review; do not auto-fail', count, 'WARNING' if count else 'PASS', f'IQR bounds: {lower:.2f} to {upper:.2f}. Statistical flags are not proof of errors.')

display(pd.DataFrame(financial_summary).pivot(index='Measure', columns='Dataset', values='Total').reindex(financial_columns))
display(pd.Series(iqr_counts, name='Rows beyond 1.5xIQR'))
flagged_ids = clean_df.loc[iqr_flagged_rows, 'Transaction_ID'].tolist()
print(f'Rows flagged in one or more screened measures: {len(flagged_ids)} of {len(clean_df)}')
print('Sample flagged transaction IDs:', flagged_ids[:15])
add_result('Suspicious Values', 'Unique rows with one or more 1.5xIQR flags', 'Report for review; not automatically an error', len(flagged_ids), 'WARNING' if flagged_ids else 'PASS', f'Screened measures: {iqr_counts}')

for name, frame in datasets.items():
    other_frame = fixed_df if name == 'Cleaned' else clean_df
    for column in financial_columns:
        total = round(float(frame[column].sum()), 2)
        other_total = round(float(other_frame[column].sum()), 2)
        totals_match = bool(np.isclose(total, other_total, atol=0.01, rtol=0))
        actual_total = f'{total:,.2f}; other CSV={other_total:,.2f}'
        add_result('Financial Totals', f'{name}: total {column}', 'Same calculated total in both CSVs', actual_total, 'PASS' if totals_match else 'FAIL')

valid_order_status = {'Cancelled', 'Completed', 'Pending', 'Returned', 'Unknown'}
unexpected_statuses = sorted(set(clean_df['Order_Status'].dropna()) - valid_order_status)
add_result('Unexpected Values', 'Order_Status allowed values', sorted(valid_order_status), sorted(clean_df['Order_Status'].dropna().unique()), 'PASS' if not unexpected_statuses else 'FAIL')
add_result('Unexpected Values', 'Unexpected Order_Status values', 'None', unexpected_statuses, 'PASS' if not unexpected_statuses else 'FAIL')

profile_columns = ['Product', 'Category', 'Customer_Segment', 'Region', 'City', 'Order_Status', 'Profit_Status']
category_profile = pd.DataFrame([
    {'Column': column, 'Observed values': sorted(clean_df[column].dropna().astype(str).unique().tolist())}
    for column in profile_columns
])
display(category_profile)
blank_labels = {
    column: int(clean_df[column].astype('string').str.strip().eq('').sum())
    for column in profile_columns
}
blank_label_count = sum(blank_labels.values())
add_result('Unexpected Values', 'Blank categorical labels', '0 blank labels', blank_labels, 'PASS' if blank_label_count == 0 else 'FAIL', 'No external product/customer/location master list was supplied; observed levels are profiled above.')

Dataset,Cleaned,Date-Fixed
Measure,,
Revenue,61039689.51,61039689.51
Cost,39438335.24,39438335.24
Operating_Expense,7682978.67,7682978.67
Gross_Profit,21601354.27,21601354.27
Net_Profit,13918375.60,13918375.60


Revenue              55
Cost                 53
Operating_Expense    71
Gross_Profit         62
Net_Profit           67
Quantity             56
Name: Rows beyond 1.5xIQR, dtype: int64

Rows flagged in one or more screened measures: 129 of 850
Sample flagged transaction IDs: ['TXN100326', 'TXN100614', 'TXN100688', 'TXN100829', 'TXN100270', 'TXN100519', 'TXN100682', 'TXN100026', 'TXN100165', 'TXN100615', 'TXN100802', 'TXN100306', 'TXN100327', 'TXN100552', 'TXN100338']


,Column,Observed values
0,Product,"[Backpack, Desk, Headphones, Keyboard, Laptop,..."
1,Category,"[Accessories, Electronics, Furniture]"
2,Customer_Segment,"[Consumer, Corporate, Small Business]"
3,Region,"[East, North, South, West]"
4,City,"[Ahmedabad, Bengaluru, Chennai, Delhi, Hyderab..."
5,Order_Status,"[Cancelled, Completed, Pending, Returned, Unkn..."
6,Profit_Status,[Profit]


## 9. Dataset Comparison

Align rows by `Transaction_ID` before comparing them. All fields are expected to match except the raw `Date` strings, which should be equal after parsing.

In [8]:
same_shape = clean_df.shape == fixed_df.shape
add_result('Dataset Comparison', 'Same record and column counts', 'Both files have 850 rows and 19 columns', f'Cleaned={clean_df.shape}; Date-Fixed={fixed_df.shape}', 'PASS' if same_shape else 'FAIL')
same_columns = clean_df.columns.tolist() == fixed_df.columns.tolist()
add_result('Dataset Comparison', 'Same column names and order', 'Identical columns in identical order', same_columns, 'PASS' if same_columns else 'FAIL')

clean_aligned = clean_df.set_index('Transaction_ID').sort_index()
fixed_aligned = fixed_df.set_index('Transaction_ID').sort_index()
same_transaction_ids = clean_aligned.index.equals(fixed_aligned.index)
add_result('Dataset Comparison', 'Same Transaction_ID values', 'Same 850 unique IDs', same_transaction_ids, 'PASS' if same_transaction_ids else 'FAIL')

comparison_groups = {
    'financial values': ['Revenue', 'Cost', 'Operating_Expense', 'Gross_Profit', 'Net_Profit', 'Profit_Status'],
    'customer information': ['Customer_ID', 'Customer_Segment'],
    'product information': ['Product', 'Category'],
    'regional information': ['Region', 'City'],
    'quantities and order statuses': ['Quantity', 'Order_Status'],
    'derived date labels': ['Month', 'Quarter', 'Year'],
}
for label, columns in comparison_groups.items():
    same_values = same_transaction_ids and np.array_equal(
        clean_aligned[columns].to_numpy(), fixed_aligned[columns].to_numpy()
    )
    add_result('Dataset Comparison', f'Same {label}', 'No differences', same_values, 'PASS' if same_values else 'FAIL')

all_non_date_columns = [column for column in expected_columns if column not in ['Date', 'Transaction_ID']]
all_non_date_same = same_transaction_ids and np.array_equal(
    clean_aligned[all_non_date_columns].to_numpy(), fixed_aligned[all_non_date_columns].to_numpy()
)
add_result('Dataset Comparison', 'All non-Date record values', 'No differences across all non-Date columns', all_non_date_same, 'PASS' if all_non_date_same else 'FAIL')

date_comparison = pd.DataFrame({
    'Cleaned parsed date': clean_by_id,
    'Date-Fixed parsed date': fixed_by_id,
}).head(5)
display(date_comparison)

,Cleaned parsed date,Date-Fixed parsed date
Transaction_ID,,
TXN100001,2025-04-13,2025-04-13
TXN100002,2025-12-15,2025-12-15
TXN100003,2025-09-28,2025-09-28
TXN100004,2025-04-17,2025-04-17
TXN100005,2025-03-13,2025-03-13


## 10. Week 1 Notebook Consistency

Cell 8 of the Week 1 notebook (zero-based cell index 7; cell ID `fd3eb9c4`) asserts that the pre-export DataFrame contains two `TXN100505` rows with quantities 3 and 75, then removes the quantity-75 row. Confirm this against the Week 1 raw input and the two final CSV outputs.

In [9]:
week1_notebook = json.loads(week1_notebook_path.read_text(encoding='utf-8'))
target_cell_matches = [
    (index, cell) for index, cell in enumerate(week1_notebook['cells'])
    if "assert set(target_records['Quantity']) == {3, 75}" in ''.join(cell.get('source', []))
]
if len(target_cell_matches) != 1:
    raise ValueError(f'Expected exactly one Week 1 cell with the TXN100505 quantity assertion; found {len(target_cell_matches)}.')
cell_index, target_cell = target_cell_matches[0]
target_id = 'TXN100505'
raw_target = raw_df.loc[raw_df['Transaction_ID'].astype('string').str.strip().eq(target_id)].copy()
raw_quantities = sorted(pd.to_numeric(raw_target['Quantity']).astype(int).tolist())
raw_other_values_match = len(raw_target) == 2 and bool(raw_target.drop(columns='Quantity').nunique().le(1).all())

output_target_summary = []
for name, frame in datasets.items():
    target = frame.loc[frame['Transaction_ID'].eq(target_id)]
    output_target_summary.append({
        'Dataset': name,
        'TXN100505 rows': len(target),
        'Quantity values': sorted(target['Quantity'].astype(int).tolist()),
    })
display(pd.DataFrame(output_target_summary))

notebook_expected_behavior = (
    len(raw_target) == 2
    and raw_quantities == [3, 75]
    and raw_other_values_match
    and all(
        len(frame.loc[frame['Transaction_ID'].eq(target_id)]) == 1
        and frame.loc[frame['Transaction_ID'].eq(target_id), 'Quantity'].iloc[0] == 3
        for frame in datasets.values()
    )
)
actual_note = (
    f'Week 1 cell {cell_index + 1} (index {cell_index}, ID {target_cell.get("id")}) expects raw quantities [3, 75]; '
    f'raw source has {len(raw_target)} rows with quantities {raw_quantities}; '
    'each supplied final CSV has one row with quantity 3.'
)
add_result('Week 1 Consistency', 'TXN100505 preprocessing expectation', 'Raw input has quantities 3 and 75; final outputs retain quantity 3 only', actual_note, 'PASS' if notebook_expected_behavior else 'FAIL', 'This is a stage-of-processing distinction, not an outdated assumption: the Week 1 notebook reads the raw dataset and exports the cleaned dataset after removing quantity 75.')
print(actual_note)
print('Interpretation: the apparent discrepancy is explained by comparing a pre-cleaning notebook assertion with post-cleaning CSV outputs; the assumption matches the raw source.')

,Dataset,TXN100505 rows,Quantity values
0,Cleaned,1,[3]
1,Date-Fixed,1,[3]


Week 1 cell 8 (index 7, ID fd3eb9c4) expects raw quantities [3, 75]; raw source has 2 rows with quantities [3, 75]; each supplied final CSV has one row with quantity 3.
Interpretation: the apparent discrepancy is explained by comparing a pre-cleaning notebook assertion with post-cleaning CSV outputs; the assumption matches the raw source.


## 11. Final Validation Summary

The result table preserves failures and warnings separately. Running this final cell exports `validation_results.csv` to the requested `outputs` folder. No source dataset, Week 1 notebook, database, or dashboard is written.

In [10]:
results_df = pd.DataFrame(results, columns=[
    'Validation Category', 'Validation Check', 'Expected Result',
    'Actual Result', 'Status', 'Remarks',
])
status_summary = results_df['Status'].value_counts().reindex(['PASS', 'FAIL', 'WARNING'], fill_value=0)
display(status_summary.rename('Checks').to_frame())
display(results_df)

output_dir = project_root / 'data' / 'validation' / 'week8'
output_dir.mkdir(parents=True, exist_ok=True)
results_path = output_dir / 'validation_results.csv'
results_df.to_csv(results_path, index=False)
print(f'Validation results exported to: {results_path}')

fail_count = int(results_df['Status'].eq('FAIL').sum())
warning_count = int(results_df['Status'].eq('WARNING').sum())
if fail_count:
    print(f'FINAL STATUS: FAIL ({fail_count} failed checks; {warning_count} warnings).')
elif warning_count:
    print(f'FINAL STATUS: PASS WITH WARNINGS ({warning_count} review items; no failed checks).')
else:
    print('FINAL STATUS: PASS (no failed checks or warnings).')

,Checks
Status,
PASS,61
FAIL,0
WARNING,8


,Validation Category,Validation Check,Expected Result,Actual Result,Status,Remarks
0,Structure,Cleaned: row and column count,850 rows and 19 columns,"(850, 19)",PASS,
1,Structure,Cleaned: column names and order,"[Transaction_ID, Date, Revenue, Cost, Operatin...","['Transaction_ID', 'Date', 'Revenue', 'Cost', ...",PASS,
2,Structure,Date-Fixed: row and column count,850 rows and 19 columns,"(850, 19)",PASS,
3,Structure,Date-Fixed: column names and order,"[Transaction_ID, Date, Revenue, Cost, Operatin...","['Transaction_ID', 'Date', 'Revenue', 'Cost', ...",PASS,
4,Missing Values,Cleaned: missing cells,0 missing cells,0,PASS,
...,...,...,...,...,...,...
64,Dataset Comparison,Same regional information,No differences,True,PASS,
65,Dataset Comparison,Same quantities and order statuses,No differences,True,PASS,
66,Dataset Comparison,Same derived date labels,No differences,True,PASS,
67,Dataset Comparison,All non-Date record values,No differences across all non-Date columns,True,PASS,


Validation results exported to: C:\Users\dell\OneDrive\Desktop\CS_Vision_Internship\data\validation\week8\validation_results.csv
FINAL STATUS: PASS WITH WARNINGS (8 review items; no failed checks).
